In [ ]:
DROP TABLE IF EXISTS Gold.FX
CREATE TABLE Gold.FX
(
    FXID INT NOT NULL,
    CurrencyFromTo VARCHAR(20) NOT NULL,
    CurrencyDate DATE NOT NULL,
    CurrencyValue DECIMAL(10,4) NOT NULL,
    IsWorkDay BIT
);
INSERT INTO Gold.FX (FXID, CurrencyFromTo, CurrencyDate, CurrencyValue, IsWorkDay)
SELECT
    ROW_NUMBER() OVER (ORDER BY Date) AS FXID,
    CurrencyConvert,
    Date,
    CurrencyValue,
    CASE 
        WHEN DATENAME(WEEKDAY, Date) IN ('Saturday', 'Sunday') THEN 0
        ELSE 1 
    END AS IsWorkDay
FROM LakehouseSilver.Silver.fx;

In [ ]:
DROP TABLE IF EXISTS Gold.WB
CREATE TABLE Gold.WB
(
    WBID INT NOT NULL,
    DateYear INT NOT NULL,
    Income BIGINT NOT NULL,
    CountryISO VARCHAR(3),
    CountryName VARCHAR(50),
    Indicator VARCHAR(50)
);
INSERT INTO Gold.WB (WBID, DateYear, Income, CountryISO, CountryName, Indicator)
SELECT
    ROW_NUMBER() OVER (ORDER BY Year) AS WBID,
    Year,
    Income,
    CountryID,
    CountryValue,
    Indicator
FROM LakehouseSilver.Silver.wb;

In [ ]:
DROP TABLE IF EXISTS Gold.Dates
CREATE TABLE Gold.Dates
(
    DateID INT NOT NULL,
    Date DATE,
    Year SMALLINT,
    Interval SMALLINT,
    IntervalName CHAR(7),
    Month SMALLINT,
    MonthName VARCHAR(15),
    YearMonth CHAR(7),
    Day SMALLINT,
    DayOfWeek SMALLINT,
    DayName VARCHAR(10),
    IsWorkDay BIT,
    IsWeekend BIT
)
WITH DateGenerator AS (
    SELECT 
        DATEADD(DAY, value, CAST('1999-01-01' AS DATE)) AS FullDate
    FROM GENERATE_SERIES(0, DATEDIFF(DAY, '1999-01-01', '2026-12-31'), 1)
)
INSERT INTO Warehouse.Gold.Dates (DateID, Date, Year, Interval, IntervalName, Month, MonthName, YearMonth, Day, DayOfWeek, DayName, IsWorkDay, IsWeekend)
SELECT
    CAST(FORMAT(FullDate, 'yyyyMMdd') AS INT),
    FullDate,
    CAST(YEAR(FullDate) AS SMALLINT),
    CAST(DATEPART(QUARTER, FullDate) AS SMALLINT),
    CAST(CONCAT('Q', DATEPART(QUARTER, FullDate), ' ', DATEPART(YEAR, FullDate)) AS CHAR(7)),
    CAST(MONTH(FullDate) AS SMALLINT),
    CAST(DATENAME(MONTH, FullDate) AS VARCHAR(15)),
    CAST(FORMAT(FullDate, 'yyyy-MM') AS CHAR(7)),
    CAST(DAY(FullDate) AS TINYINT),
    CAST(DATEPART(WEEKDAY, FullDate) AS TINYINT),
    CAST(DATENAME(WEEKDAY, FullDate) AS VARCHAR(15)),
    CASE WHEN DATEPART(WEEKDAY, FullDate) IN (1, 7) THEN 0 ELSE 1 END,
    CASE WHEN DATEPART(WEEKDAY, FullDate) IN (1, 7) THEN 1 ELSE 0 END
FROM DateGenerator;


In [ ]:
DROP TABLE IF EXISTS Gold.Zones
CREATE TABLE Gold.Zones
(
    ZoneID INT NOT NULL,
    LocationID INT NOT NULL,
    Neighborhood VARCHAR(15),
    ZoneName VARCHAR(60) NOT NULL,
    ServiceZone VARCHAR(15),
    IsAirport BIT
)
INSERT INTO Gold.Zones (ZoneID, LocationID, Neighborhood, ZoneName, ServiceZone, IsAirport)
SELECT
    LocationID,
    LocationID,
    Neighborhood,
    ZoneName,
    ServiceZone,
    IsAirport
FROM LakehouseSilver.Silver.zones

In [ ]:
DROP TABLE IF EXISTS Gold.FactAirQualityDaily
CREATE TABLE Gold.FactAirQualityDaily
(
	AQID BIGINT NOT NULL,
	Value DECIMAL(6, 3) NOT NULL,
	MeasureDateTime DATETIME2(0) NOT NULL,
	MeasureName VARCHAR(10),
	MeasureUnit VARCHAR(10),
	MeasureSensorID INT,
	MeasureLocation VARCHAR(15)
)
INSERT INTO Gold.FactAirQualityDaily (AQID, Value, MeasureDateTime, MeasureName, MeasureUnit, MeasureSensorID, MeasureLocation)
SELECT
    HASHBYTES('SHA2_256', CONCAT(MeasureDateTime, MeasureValue)) AS AQID,
    MeasureValue,
    MeasureDateTime,
    MeasureName,
    Unit,
    SensorID,
    MeasureLocation
FROM LakehouseSilver.Silver.openaq

In [ ]:
DROP TABLE IF EXISTS Gold.FactTaxiDaily
CREATE TABLE Gold.FactTaxiDaily
(
    FareID BIGINT NOT NULL,
    PickupDate DATETIME2(0) NOT NULL,
    PickLocation INT NOT NULL,
    DropDate DATETIME2(0) NOT NULL,
    DropLocation INT NOT NULL,
    Distance DECIMAL(8,3),
    Fare DECIMAL(8,2) NOT NULL,
    Passengers INT,
    TaxiType VARCHAR(7)
);
INSERT INTO Gold.FactTaxiDaily (FareID, PickupDate, PickLocation, DropDate, DropLocation, Distance, Fare, Passengers, TaxiType)
SELECT
    HASHBYTES('SHA2_256', CONCAT(PickDate, PickLocation, DropLocation, DropLocation, Fare)) AS FareID,
    PickDate,
    PickLocation,
    DropDate,
    DropLocation,
    Distance,
    COALESCE(Fare, 0),
    Passengers,
    TLCType
FROM LakehouseSilver.Silver.tlc

In [ ]:
DROP TABLE IF EXISTS Gold.AggDailyAirQuality
CREATE TABLE Gold.AggDailyAirQuality AS
SELECT
    CAST(MeasureDateTime AS DATE) AS Date,
    MeasureLocation,
    MeasureName,
    COALESCE(AVG(Value), 0) AS MeasureValue
FROM Gold.FactAirQualityDaily
GROUP BY CAST(MeasureDateTime AS DATE), MeasureLocation, MeasureName

In [ ]:
DROP TABLE IF EXISTS Gold.AggDailyTaxi
CREATE TABLE Gold.AggDailyTaxi AS
SELECT
    CONVERT(DATE, PickupDate) AS Date,
    TaxiType,
    SUM(Fare) AS FareSUM,
    COALESCE(AVG(NULLIF(Fare, 0)),0) AS FareAVG,
    COUNT(*) AS FareCOUNT,
    SUM(Distance) AS DistanceSUM,
    COALESCE(AVG(CASE WHEN Distance = 0 AND Fare = 0 THEN NULL ELSE Distance END), 0) AS DistanceAVG,
    SUM(Passengers) AS PassengersSUM,
    ROUND(COALESCE(AVG(NULLIF(CAST(Passengers AS FLOAT), 0)), 0), 3) AS PassengersAVG,
    DropLocation,
    PickLocation
FROM Gold.FactTaxiDaily
GROUP BY CONVERT(DATE, PickupDate), TaxiType, DropLocation, PickLocation